# Persona 3: estimación de bacterias con Kalman

Queremos estimar las cantidades sensibles (S), tolerantes (T) y resistentes (R) a partir de mediciones imperfectas. La verdad para este experimento se obtiene del Euler del equipo.

## 1. Importar funciones

In [1]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
raiz = Path.cwd()
if not (raiz/'src').is_dir():
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
from src.euler_model import load_params, simulate
from src.filtro_kalman import generar_mediciones, filtrar, rmse, graficar, ejecutar_experimento


## 2. Obtener la trayectoria real
El estado es $x=[S,T,R]^T$. Euler entrega la concentración aplicada en cada paso. Tiempo en días y población en UFC/mL.

In [2]:
parametros, escenario = load_params('D')
t, verdad, C, dt = simulate('D', parametros, escenario)
print('Estado inicial:', verdad[0])
print('Estado final:', verdad[-1])


Estado inicial: [1.e+07 1.e+03 1.e+01]
Estado final: [1.03286603e+03 4.45532166e+07 6.66297718e+06]


## 3. Simular mediciones
Empezamos midiendo S, T y R, así que $H=I$. Las observaciones tienen ruido gaussiano y una semilla fija. La matriz `R` expresa la varianza de cada sensor.

In [3]:
H = np.eye(3)
medidas, R = generar_mediciones(verdad, H, ruido=0.10, semilla=0)
print('H:', H, sep='\n')
print('R inicial:', R[0], sep='\n')
print('Verdad inicial:', verdad[0])
print('Medición inicial:', medidas[0])


H:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
R inicial:
[[1.e+12 0.e+00 0.e+00]
 [0.e+00 1.e+04 0.e+00]
 [0.e+00 0.e+00 1.e+00]]
Verdad inicial: [1.e+07 1.e+03 1.e+01]
Medición inicial: [1.01257302e+07 9.86789514e+02 1.06404227e+01]


## 4. Aplicar el EKF
La predicción usa Euler; el jacobiano aproxima cómo cambian los siguientes valores. $P$ mide la incertidumbre de la estimación y $Q$ la de la predicción. La corrección combina predicción y observación con la ganancia de Kalman. El filtro recibe las mediciones, no los valores reales posteriores.

In [4]:
x0 = verdad[0] * np.array([1.2, 0.8, 1.3])
P0 = np.diag((0.5 * np.maximum(x0, 1.0))**2)
estimados, P = filtrar(t, medidas, R, H, parametros, C, x0, P0, q_rel=0.02)
resultado = {'t':t, 'verdad':verdad, 'medidas':medidas, 'estimados':estimados, 'P':P, 'H':H}


## 5. Ver la comparación y calcular RMSE
Las gráficas muestran valor real, medición y EKF. La banda representa incertidumbre gaussiana aproximada del 95%. El RMSE está en UFC/mL y debe compararse dentro de cada población.

In [5]:
figura = graficar(resultado)
display(figura)
plt.close(figura)
print('RMSE medido [S,T,R]:', rmse(verdad, medidas))
print('RMSE EKF    [S,T,R]:', rmse(verdad, estimados))


<Figure size 1500x450 with 3 Axes>

RMSE medido [S,T,R]: [ 183659.35588659 1285035.81006556  162781.83797874]
RMSE EKF    [S,T,R]: [ 54964.83465116 349222.71870678  50214.57524564]


## 6. Efecto de aumentar el ruido
Repetimos el mismo caso con 5%, 10% y 20% de ruido. El filtro usa la varianza de cada medición. Estos datos son sintéticos y usan el mismo modelo para simular y predecir.

In [6]:
for ruido in (0.05, 0.10, 0.20):
    r = ejecutar_experimento(ruido=ruido)
    print(f'Ruido {ruido:.0%}: RMSE EKF [S,T,R] = {rmse(r["verdad"], r["estimados"])}')


Ruido 5%: RMSE EKF [S,T,R] = [ 37396.67175168 264845.95914891  34259.08720246]
Ruido 10%: RMSE EKF [S,T,R] = [ 54964.83465116 349222.71870678  50214.57524564]
Ruido 20%: RMSE EKF [S,T,R] = [ 83806.39369145 454726.99152256  68838.4109278 ]


## 7. Reto: estimar T sin medirla
Ahora H observa solo S y R. No existe medición de T para comparar; sí podemos evaluar su estimación frente a la verdad sintética de Euler.

In [7]:
reto = ejecutar_experimento(medir_T=False)
print('H del reto:', reto['H'], sep='\n')
print('RMSE EKF [S,T,R]:', rmse(reto['verdad'], reto['estimados']))
figura = graficar(reto)
display(figura)
plt.close(figura)


H del reto:
[[1. 0. 0.]
 [0. 0. 1.]]
RMSE EKF [S,T,R]: [ 56815.03758262 613965.26820153  52740.79945055]


<Figure size 1500x450 with 3 Axes>

## 8. Conclusiones y límites

- Comparar los RMSE para saber si el EKF mejora las mediciones en cada población.
- El ruido, Q, R y el estado inicial son supuestos ilustrativos; con datos reales hay que calibrarlos.
- El caso sin medir T es más difícil: una gráfica cercana no prueba que T sea identificable.
- Euler aún tiene un desfase en el calendario de concentración. Repetir el análisis cuando se corrija.
- Este trabajo estima poblaciones; la integración con Bayes y las decisiones de dosis son tareas del equipo.